In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold, mutual_info_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova
import itertools
from statsmodels.stats.multitest import multipletests
from skbio.stats.distance import permdisp

## Data preparation

In [2]:
dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Beeswarm/Significance.xlsx"
df = pd.read_excel(dataset_total_path, sheet_name="Limo")

In [3]:
y = df['Class']
X = df.drop('Class', axis = 1)

In [4]:
X.head()

,Glucosyl-limonin,Deoxylimononic acid D-ring-lactone,Rutaevin,Salannin,Nomilin,Limonoate,Fraxinellone,Obacunoic acid,"Methyl (2R)-[(1S,3S,7R,8R,12S,13S)-13-(3-furyl)-6,6,8,12-tetramethyl-17-methylene-5,15-dioxo-2,14-dioxatetracyclo[7,7,1,01,12 03,8]heptadec-7-yl](hydroxy)acetate",Limonin,Obacunone
0,6.580088e+06,8.052531e+06,323616.239017,1.522009e+07,3.901699e+07,3.423551e+07,6.193731e+07,2.804679e+07,5.702071e+07,3.091842e+08,6.409729e+08
1,6.424336e+06,7.597426e+06,126669.424524,1.134110e+07,3.751774e+07,3.139880e+07,6.489341e+07,2.714036e+07,7.251755e+07,2.707856e+08,4.688408e+08
2,6.055534e+06,8.569100e+06,155772.762872,1.568066e+07,4.437806e+07,2.895881e+07,6.663376e+07,4.051424e+07,6.884355e+07,3.791600e+08,6.744009e+08
3,6.670318e+06,1.030463e+07,201945.387789,1.331600e+07,4.598107e+07,3.171062e+07,7.056140e+07,3.578962e+07,6.990944e+07,4.031859e+08,7.932977e+08
4,5.561220e+06,1.105310e+07,386219.381390,1.295076e+07,4.221679e+07,4.019129e+07,6.061064e+07,4.761629e+07,7.424297e+07,4.517089e+08,9.031800e+08


### Log2 transformation

In [5]:
print("var before log2 transormation : " , X.var(axis=0).mean())
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X = log2_transformer.fit_transform(X)
print("var after log2 transormation : " , X.var(axis=0).mean())

var before log2 transormation :  3.880996696052931e+16
var after log2 transormation :  8.273331527693427


### Standardization

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
#X = X[:12, :]
#y = y.iloc[:12]

## Global permanova

In [8]:
# Compute pairwise Euclidean distances
dist_array = pdist(X, metric="euclidean")
dist_matrix = squareform(dist_array)

# Create DistanceMatrix object
sample_ids = [f"S{i+1:02d}" for i in range(X.shape[0])]
dm = DistanceMatrix(dist_matrix, ids=sample_ids)

res = permanova(
    distance_matrix=dm,
    grouping=y.values,
    permutations=50000
)

print(res)

method name               PERMANOVA
test statistic name        pseudo-F
sample size                      24
number of groups                  4
test statistic            65.373206
p-value                     0.00002
number of permutations        50000
Name: PERMANOVA results, dtype: object


## Pairwise permanova

In [9]:
dist_array = pdist(X, metric="euclidean")
dist_matrix = squareform(dist_array)

sample_ids = [f"S{i+1:02d}" for i in range(len(y))]
dm_full = DistanceMatrix(dist_matrix, ids=sample_ids)

In [10]:
classes = y.unique()
results = []

for c1, c2 in itertools.combinations(classes, 2):

    idx = y.isin([c1, c2]).values
    ids = np.array(sample_ids)[idx]

    dm_sub = dm_full.filter(ids)
    y_sub = y[idx]

    res = permanova(
        distance_matrix=dm_sub,
        grouping=y_sub.values,
        permutations=50000
    )

    results.append({
        "Group1": c1,
        "Group2": c2,
        "pseudo-F": res["test statistic"],
        "p-value": res["p-value"]
    })

pairwise_df = pd.DataFrame(results)

In [11]:
pairwise_df["p_adj"] = multipletests(
    pairwise_df["p-value"],
    method="fdr_bh"
)[1]


In [12]:
pairwise_df

,Group1,Group2,pseudo-F,p-value,p_adj
0,LN,LP,16.710408,0.0034,0.0035
1,LN,SN,65.498748,0.0031,0.0035
2,LN,SP,212.507177,0.0024,0.0035
3,LP,SN,70.910410,0.0028,0.0035
4,LP,SP,175.381150,0.0022,0.0035
5,SN,SP,4.489950,0.0035,0.0035


In [13]:
res_disp = permdisp(dm, grouping=y.values, permutations=50000)
print("PERMDISP p-value:", res_disp["p-value"])

C:\Users\tamer\anaconda3\Lib\site-packages\skbio\stats\ordination\_principal_coordinate_analysis.py:157: RuntimeWarning: EIGH: since no value for dimensions is specified, PCoA for all dimensions will be computed, which may result in long computation time if the original distance matrix is large.
  warn(


PERMDISP p-value: 0.051658966820663586
